In [1]:
# =========================================
# MODEL 1 TRAINING - CATEGORY CLASSIFIER
# =========================================

import os
import json
import time
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
import torchvision.transforms as T
from sklearn.metrics import confusion_matrix, classification_report
import timm
from tqdm import tqdm

In [2]:
# from pathlib import Path

# root = Path("/kaggle/input")
# for p in root.rglob("*"):
#     if p.is_dir():
#         print(p)

In [3]:
# -----------------------------------------
# CONFIG
# -----------------------------------------
SEED = 42

# CHANGE THIS TO YOUR REAL KAGGLE DATASET PATH
DATA_ROOT = "/kaggle/input/datasets/rupeshbhardwaj420/category-dataset-model-1/category_model_dataset"

OUT_DIR = Path("/kaggle/working/model1_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 260
BATCH_SIZE = 64
EPOCHS = 8
LR = 3e-4
WEIGHT_DECAY = 1e-4
MODEL_NAME = "efficientnet_b2"   # good first baseline
VAL_RATIO = 0.2
NUM_WORKERS = 2

In [4]:
# -----------------------------------------
# REPRODUCIBILITY
# -----------------------------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [5]:
# -----------------------------------------
# TRANSFORMS
# -----------------------------------------
train_tf = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
])

val_tf = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
])


In [6]:
# -----------------------------------------
# DATASET LOADING
# -----------------------------------------
full_train = ImageFolder(DATA_ROOT, transform=train_tf)
full_val = ImageFolder(DATA_ROOT, transform=val_tf)

n = len(full_train)
idx = np.arange(n)
np.random.shuffle(idx)

val_n = int(n * VAL_RATIO)
val_idx = idx[:val_n]
train_idx = idx[val_n:]

train_ds = Subset(full_train, train_idx)
val_ds = Subset(full_val, val_idx)

class_to_idx = full_train.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}
num_classes = len(class_to_idx)

print("Total images:", n)
print("Train images:", len(train_ds))
print("Val images:", len(val_ds))
print("Num classes:", num_classes)
print("Class names:", list(class_to_idx.keys()))

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

Total images: 17965
Train images: 14372
Val images: 3593
Num classes: 10
Class names: ['Animal', 'flags', 'flower', 'food', 'fruits_and_vegetables', 'monument', 'shapes', 'sports_equipments', 'vehicle', 'weather']


In [7]:

# -----------------------------------------
# MODEL
# -----------------------------------------
model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=num_classes
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)


model.safetensors:   0%|          | 0.00/36.8M [00:00<?, ?B/s]

In [8]:

# -----------------------------------------
# EVALUATION FUNCTION
# -----------------------------------------
@torch.no_grad()
def evaluate():
    model.eval()

    y_true = []
    y_pred = []
    top3_correct = 0
    total = 0
    running_val_loss = 0.0

    for x, y in val_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        pred1 = torch.argmax(logits, dim=1)
        top3 = torch.topk(logits, k=min(3, num_classes), dim=1).indices

        top3_correct += (top3 == y.unsqueeze(1)).any(dim=1).sum().item()
        total += y.size(0)
        running_val_loss += loss.item() * x.size(0)
        y_true.extend(y.cpu().tolist())
        y_pred.extend(pred1.cpu().tolist())

    top1 = (np.array(y_true) == np.array(y_pred)).mean()
    top3_acc = top3_correct / max(total, 1)
    avg_val_loss = running_val_loss / max(total, 1)
    cm = confusion_matrix(y_true, y_pred)

    return float(avg_val_loss), float(top1), float(top3_acc), cm, y_true, y_pred

In [9]:

# -----------------------------------------
# TRAINING LOOP
# -----------------------------------------
best_top1 = -1.0
best_path = OUT_DIR / "model1_category_best.pt"

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    seen = 0
    t0 = time.time()

    for x, y in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        bs = x.size(0)
        running_loss += loss.item() * bs
        seen += bs

    train_loss = running_loss / max(seen, 1)
    val_loss, top1, top3, cm, y_true, y_pred = evaluate()
    dt = time.time() - t0

    print(
        f"Epoch {epoch}: "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"top1={top1:.4f} | "
        f"top3={top3:.4f} | "
        f"time={dt:.1f}s"
    )

    if top1 > best_top1:
        best_top1 = top1
        torch.save(model.state_dict(), best_path)
        print("✅ Saved best model:", best_path)

epoch 1/8:  30%|███       | 68/225 [00:58<01:58,  1.32it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
epoch 1/8: 100%|██████████| 225/225 [03:06<00:00,  1.21it/s]
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1: train_loss=0.1563 | val_loss=0.0556 | top1=0.9833 | top3=0.9986 | time=215.2s
✅ Saved best model: /kaggle/working/model1_outputs/model1_category_best.pt


epoch 2/8:  12%|█▏        | 26/225 [00:21<02:05,  1.59it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
epoch 2/8:  43%|████▎     | 96/225 [01:13<01:20,  1.60it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
epoch 2/8: 100%|██████████| 225/225 [02:48<00:00,  1.34it/s]
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 2: train_loss=0.0279 | val_loss=0.0433 | top1=0.9861 | top3=0.9989 | time=191.2s
✅ Saved best model: /kaggle/working/model1_outputs/model1_category_best.pt


epoch 3/8:   6%|▌         | 14/225 [00:10<02:17,  1.54it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
epoch 3/8:  12%|█▏        | 27/225 [00:20<02:27,  1.34it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
epoch 3/8: 100%|██████████| 225/225 [02:42<00:00,  1.38it/s]
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


...

Epoch 6: train_loss=0.0138 | val_loss=0.0568 | top1=0.9864 | top3=0.9986 | time=186.5s


epoch 7/8:  55%|█████▍    | 123/225 [01:30<01:09,  1.47it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
epoch 7/8:  68%|██████▊   | 152/225 [01:53<00:57,  1.27it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
epoch 7/8: 100%|██████████| 225/225 [02:44<00:00,  1.37it/s]
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 7: train_loss=0.0237 | val_loss=0.0604 | top1=0.9886 | top3=0.9983 | time=187.5s


epoch 8/8:   5%|▌         | 12/225 [00:09<02:46,  1.28it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
epoch 8/8:  60%|██████    | 136/225 [01:42<01:05,  1.37it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
epoch 8/8: 100%|██████████| 225/225 [02:46<00:00,  1.35it/s]
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 8: train_loss=0.0325 | val_loss=0.0465 | top1=0.9858 | top3=0.9983 | time=189.6s


In [10]:
# -----------------------------------------
# SAVE LAST MODEL
# -----------------------------------------
last_path = OUT_DIR / "model1_category_last.pt"
torch.save(model.state_dict(), last_path)

# -----------------------------------------
# SAVE LABEL MAP
# -----------------------------------------
(OUT_DIR / "class_to_idx.json").write_text(
    json.dumps(class_to_idx, indent=2)
)

179

In [11]:
# -----------------------------------------
# SAVE METRICS
# -----------------------------------------
metrics = {
    "best_val_top1": best_top1,
    "model_name": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "num_classes": num_classes
}
(OUT_DIR / "metrics.json").write_text(
    json.dumps(metrics, indent=2)
)


153

In [12]:
# -----------------------------------------
# SAVE CONFIG SNAPSHOT
# -----------------------------------------
config_snapshot = {
    "seed": SEED,
    "data_root": DATA_ROOT,
    "val_ratio": VAL_RATIO,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "model_name": MODEL_NAME,
    "num_classes": num_classes
}
(OUT_DIR / "config_model1.json").write_text(
    json.dumps(config_snapshot, indent=2)
)


297

In [13]:

# -----------------------------------------
# SAVE CLASSIFICATION REPORT
# -----------------------------------------
report = classification_report(
    y_true,
    y_pred,
    target_names=[idx_to_class[i] for i in range(num_classes)],
    output_dict=True
)

(OUT_DIR / "classification_report.json").write_text(
    json.dumps(report, indent=2)
)

print("\n✅ DONE. Exported files:")
for p in OUT_DIR.iterdir():
    print(" -", p)


✅ DONE. Exported files:
 - /kaggle/working/model1_outputs/class_to_idx.json
 - /kaggle/working/model1_outputs/classification_report.json
 - /kaggle/working/model1_outputs/model1_category_best.pt
 - /kaggle/working/model1_outputs/metrics.json
 - /kaggle/working/model1_outputs/config_model1.json
 - /kaggle/working/model1_outputs/model1_category_last.pt
